In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, END

from pydantic import BaseModel, Field
from typing import Annotated, Optional, Any
import operator
from dotenv import load_dotenv

In [2]:
load_dotenv("../.env")

True

### 데이터 구조 정의

In [3]:
# 페르소나를 나타내는 데이터 모델
class Persona(BaseModel):
    name: str = Field(..., description="페르소나의 이름")
    background: str = Field(..., description="페르소나의 배경")

# 페르소나 목록을 나타내는 데이터 모델
class Personas(BaseModel):
    personas: list[Persona] = Field(default_factory=list, description="페르소나 목록")

# 인터뷰 내용을 나타내는 데이터 모델
class Interview(BaseModel):
    persona: Persona = Field(..., description="인터뷰 대상 페르소나")
    question: str = Field(..., description="인터뷰 질문")
    answer: str = Field(..., description="인터뷰 응답")

# 인터뷰 결과 목록을 나타내는 데이터 모델
class InterviewResult(BaseModel):
    interviews: list[Interview] = Field(default_factory=list, description="인터뷰 결과 목록")

# 평가 결과를 나타내는 데이터 모델
class EvaluationResult(BaseModel):
    reason: str = Field(..., description="판단 이유")
    is_sufficient: bool = Field(..., description="정보가 충분한지 여부")

In [4]:
# 스테이트의 타입은 다음과 같이 정의
# personas와 interviews 필드는 노드에 의해 데이터가 추가되는 항목이므로 Annotated와 operator.add를 사용하여 정의

# 요구사항 정의서 생성 AI 에이전트의 스테이트
class InterviewState(BaseModel):
    user_request: str = Field(..., description="사용자 요청")
    personas: Annotated[list[Persona], operator.add] = Field(default_factory=list, description="생성된 페르소나 목록")
    interviews: Annotated[list[Interview], operator.add] = Field(default_factory=list, description="실시된 인터뷰 목록")
    requirements_doc: str = Field(default="", description="생성된 요구사항 정의서")
    iteration: int = Field(default=0, description="페르소나 생성과 인터뷰의 반복 횟수")
    is_information_sufficient: bool = Field(default=False, description="정보가 충분한지 여부")
    evaluation_reason: str = Field(default="", description="정보 충분성 판단 이유")

### PersonaGenerator

In [5]:
# 사용자의 요구에 기반하여 다양한 페르소나를 생성
# 생성할 페르소나 수는 k 매개변수로 제어할 수 있다.
class PersonaGenerator:
    def __init__(self, llm: ChatOpenAI, k: int = 5):
        self.llm = llm.with_structured_output(Personas)
        self.k = k

    def run(self, user_request: str) -> Personas:
        prompt = ChatPromptTemplate.from_messages([
            ("system",
             "당신은 사용자 인터뷰용 다양한 페르소나를 만드는 전문가입니다.",),
            ("human",
             f"다음 사용자 요청에 관한 인터뷰를 위해 {self.k}명의 다양한 페르소나를 생성해 주세요.\n\n"
             "사용자 요청: {user_request}\n\n"
             "각 페르소나에는 이름과 간단한 배경을 포함해 주세요. 연령, 성별, 직업. 기술적 전문 지식에서 다양성을 확보해 주세요.")
        ])

        chain = prompt | self.llm
        return chain.invoke({"user_request": user_request})

### InterviewConductor

In [6]:
class InterviewConductor:
    def __init__(self, llm: ChatOpenAI):
        self.llm = llm

    def run(self, user_request: str, personas: list[Persona]) -> InterviewResult:
        questions = self._generate_questions(user_request=user_request, personas=personas)
        answers = self._generate_answers(personas=personas, questions=questions)
        interviews = self._create_interviews(personas=personas, questions=questions, answers=answers)
        return InterviewResult(interviews=interviews)

    def _generate_questions(self, user_request: str, personas: list[Persona]) -> list[str]:
        question_prompt = ChatPromptTemplate.from_messages([
            ("system",
             "당신은 사용자 요구사항에 기반하여 적절한 질문을 생성하는 전문가입니다."),
            ("human",
             "다음 페르소나와 관련된 사용자 요청에 대해 하나의 질문을 생성해 주세요.\n\n"
             "사용자 요청: {user_request}\n"
             "페르소나: {persona_name} - {persona_background}\n\n"
             "질문은 구체적이며, 이 페르소나의 관점에서 중요한 정보를 끌어낼 수 있도록 설계해 주세요.")
        ])
        question_chain = question_prompt | self.llm | StrOutputParser()
        question_queries = [
            {"user_request": user_request,
             "persona_name": persona.name,
             "persona_background": persona.background}
             for persona in personas
        ]
        return question_chain.batch(question_queries)

    def _generate_answers(self, personas: list[Persona], questions: list[str]) -> list[str]:
        answer_prompt = ChatPromptTemplate.from_messages([
            ("system",
             "당신은 다음 페르소나로서 답변하고 있습니다: {persona_name} - {persona_background}"),
            ("human", "질문: {question}")
        ])
        answer_chain = answer_prompt | self.llm | StrOutputParser()
        answer_queries = [
            {"persona_name": persona.name,
             "persona_background": persona.background,
             "question": question}
             for persona, question in zip(personas, questions)
        ]
        return answer_chain.batch(answer_queries)

    def _create_interviews(self, personas: list[Persona], questions: list[str], answers: list[str]) -> list[Interview]:
        return [
            Interview(persona=persona, question=question, answer=answer)
            for persona, question, answer in zip(personas, questions, answers)
        ]

### InformationEvaluator

In [7]:
class InformationEvaluator:
    def __init__(self, llm: ChatOpenAI):
        self.llm = llm.with_structured_output(EvaluationResult)

    def run(self, user_request: str, interviews: list[Interview]) -> EvaluationResult:
        prompt = ChatPromptTemplate.from_messages([
            ("system",
             "당신은 포괄적인 요구사항 문서를 작성하기 위한 정보의 충분성을 평가하는 전문가입니다."),
            ("human",
             "다음 사용자 요청과 인터뷰 결과를 바탕으로, 포괄적인 요구사항 문서를 작성하기에 충분한 정보가 모였는지 판단해 주세요.\n\n"
             "사용자 요청: {user_request}\n\n"
             "인터뷰 결과:\n{interview_results}")
        ])

        chain = prompt | self.llm

        return chain.invoke({
            "user_request": user_request,
            "interview_results": "\n".join(
                f"페르소나: {i.persona.name} - {i.persona.background}\n"
                f"질문: {i.question}\n답변: {i.answer}\n"
                for i in interviews
            )
        })

### RequirementsDocumentGenerator

In [8]:
class RequirementsDocumentGenerator:
    def __init__(self, llm: ChatOpenAI):
        self.llm = llm

    def run(self, user_request: str, interviews: list[Interview]) -> str:
        prompt = ChatPromptTemplate.from_messages([
            ("system",
             "당신은 수집한 정보를 바탕으로 요구사항 문서를 작성하는 전문가입니다."),
            ("human",
             "다음 사용자 요청과 여러 페르소나의 인터뷰 결과를 바탕으로 요구사항 문서를 작성해 주세요.\n\n"
             "사용자 요청: {user_request}\n\n"
             "인터뷰 결과:\n{interview_results}\n"
             "요구사항 문서에는 다음 섹션을 포함해 주세요:\n"
             "1. 프로젝트 개요\n"
             "2. 주요 기능\n"
             "3. 비기능 요구사항\n"
             "4. 제약 조건\n"
             "5. 타깃 사용자\n"
             "6. 우선순위\n"
             "7. 위험과 완화 방안\n\n"
             "출력은 반드시 한국어로 부탁드립니다.\n\n 요구사항 문서:"
             )
        ])

        chain = prompt | self.llm | StrOutputParser()
        return chain.invoke({
            "user_request": user_request,
            "interview_results": "\n".join(
                f"페르소나: {i.persona.name} - {i.persona.background}\n"
                f"질문: {i.question}\n답변: {i.answer}\n"
                for i in interviews
            )
        })

### 워크플로 구축

In [9]:
class DocumentationAgent:
    def __init__(self, llm: ChatOpenAI, k: Optional[int] = 5):
        self.persona_generator = PersonaGenerator(llm=llm, k=k)
        self.interview_conductor = InterviewConductor(llm=llm)
        self.information_evaluator = InformationEvaluator(llm=llm)
        self.requirements_generator = RequirementsDocumentGenerator(llm=llm)
        self.graph = self._create_graph()

    def _create_graph(self) -> StateGraph:
        workflow = StateGraph(InterviewState)
        workflow.add_node("generate_personas", self._generate_personas)
        workflow.add_node("conduct_interviews", self._conduct_interviews)
        workflow.add_node("evaluate_information", self._evaluate_information)
        workflow.add_node("generate_requirements", self._generate_requirements)

        workflow.set_entry_point("generate_personas")
        workflow.add_edge("generate_personas", "conduct_interviews")
        workflow.add_edge("conduct_interviews", "evaluate_information")

        workflow.add_conditional_edges(
            "evaluate_information",
            lambda state: not state.is_information_sufficient and state.iteration < 5,
            {True: "generate_personas", False: "generate_requirements"}
        )
        workflow.add_edge("generate_requirements", END)
        return workflow.compile()

    def _generate_personas(self, state: InterviewState) -> dict[str, Any]:
        new_personas: Personas = self.persona_generator.run(state.user_request)
        return {
            "personas": new_personas.personas,
            "iteration": state.iteration + 1
        }

    def _conduct_interviews(self, state: InterviewState) -> dict[str, Any]:
        new_interviews: InterviewResult = self.interview_conductor.run(state.user_request, state.personas[-5:])
        return {"interviews": new_interviews.interviews}

    def _evaluate_information(self, state: InterviewState) -> dict[str, Any]:
        evaluation_result: EvaluationResult = self.information_evaluator.run(
            state.user_request, state.interviews
        )
        return {
            "is_information_sufficient": evaluation_result.is_sufficient,
            "evaluation_reason": evaluation_result.reason
        }

    def _generate_requirements(self, state: InterviewState) -> dict[str, Any]:
        requirements_doc: str = self.requirements_generator.run(
            state.user_request, state.interviews
        )
        return {"requirements_doc": requirements_doc}

    def run(self, user_request: str) -> str:
        initial_state = InterviewState(user_request=user_request)
        final_state = self.graph.invoke(initial_state)
        return final_state["requirements_doc"]

### 10.6 에이전트 실행과 결과 확인

In [10]:
from IPython.display import display, Markdown

llm = ChatOpenAI(model="gpt-6-luna", reasoning_effort="none")
agent = DocumentationAgent(llm=llm, k=5)
requirements_doc = agent.run(user_request="스마트폰용 건강 관리 앱을 개발하고 싶다")
display(Markdown(requirements_doc))

c:\Users\limux\Documents\1. Projects\llm-engineer-roadmap\02-rag-agents-langchain-langgraph\.venv\Lib\site-packages\langsmith\_internal\_addressing.py:104: LangSmithBetaWarning: Addressing runs with `langsmith.address` is in beta and is enabled per workspace. A workspace without it rejects the run, so the trace is lost rather than falling back to a project. The behavior may change without notice.
  _warn_once(


# 스마트폰 건강 관리 앱 요구사항 문서

## 1. 프로젝트 개요

### 1.1 목적
스마트폰과 웨어러블의 건강 데이터를 자동으로 모으고, 필요한 건강 정보를 간단히 기록·확인할 수 있는 모바일 앱을 개발한다. 사용자가 자신의 생활 패턴과 건강 기록을 이해하고 꾸준한 습관을 형성하도록 돕되, 앱이 의료 진단이나 치료를 대신하지 않도록 한다.

### 1.2 제품 방향
- 자동 수집을 우선하고, 기기나 연동이 지원되지 않는 경우에만 간단한 수동 입력을 제공한다.
- 건강 정보와 기록 추이를 한눈에 보여주고, 자세한 분석은 원하는 사용자가 확인할 수 있도록 한다.
- 연령, 기술 숙련도, 근무 일정이 달라도 사용할 수 있도록 설정과 알림을 유연하게 한다.
- 건강 데이터의 수집·저장·공유·삭제를 사용자가 통제할 수 있게 한다.

### 1.3 제품 범위
**초기 버전(MVP)**
- iOS HealthKit 및 Android Health Connect 연동
- 걸음 수·활동량·수면·심박수 등 지원 가능한 데이터의 자동 수집
- 혈압·혈당·복약·컨디션 등의 간편 수동 기록
- 날짜별 기록과 주간 추이 대시보드
- 사용자 조정 가능한 알림
- 데이터 동의·연동 해제·삭제 기능

**후속 확장 후보**
- 주요 웨어러블·운동·식단 서비스의 직접 연동
- 식사 사진·즐겨찾기 기록, 운동 기록 고도화
- 개인화된 목표·추세 분석, CSV 내보내기 및 문서화된 API
- 선택형 친구 챌린지와 게임형 보상

---

## 2. 주요 기능

### 2.1 온보딩 및 초기 설정
- 앱은 가입과 초기 설정을 최소 단계로 제공해야 한다.
- 권한 요청 전, 수집할 데이터 항목과 사용 목적을 쉬운 말로 설명해야 한다.
- 사용자는 건강 데이터 항목별로 연동 권한을 선택하거나 거절할 수 있어야 한다.
- 큰 글씨 모드, 기본 건강 관심사, 알림 시간 등 핵심 설정을 선택할 수 있어야 한다.
- 첫 기록이나 기기 연결을 단계별 안내로 지원하고, 설정을 나중에 변경하는 방법을 제공해야 한다.

### 2.2 건강 데이터 연동 및 자동 수집
- iOS에서는 HealthKit, Android에서는 Health Connect를 기본 연동 경로로 지원해야 한다.
- 사용자의 동의를 받은 항목만 읽어야 하며, 앱에서 데이터를 기록하는 기능이 필요한 경우 쓰기 권한을 별도로 요청해야 한다.
- MVP에서 우선 지원할 데이터는 다음과 같다.
  - 걸음 수 및 활동량
  - 운동 시간·종류(플랫폼에서 제공되는 경우)
  - 수면 시간 및 시간대(플랫폼에서 제공되는 경우)
  - 심박수(플랫폼에서 제공되는 경우)
- 후속 지원 후보로 HRV, 혈중 산소, 체온 추세, 체중·체성분, 혈압·혈당, 식사·수분·기분·복약 기록을 검토한다.
- 각 데이터에 원본 기기 또는 서비스, 측정 시각, 단위, 마지막 동기화 시점을 표시해야 한다.
- 중복 데이터가 유입되는 경우 출처와 시간 정보를 이용해 중복을 처리해야 하며, 처리 결과를 사용자가 확인할 수 있어야 한다.
- 데이터 누락, 연동 오류, 권한 철회 시 현재 상태와 해결 방법을 쉬운 말로 안내해야 한다.

### 2.3 건강 정보 수동 기록
- 사용자는 혈압을 수축기·이완기 수치로 빠르게 입력할 수 있어야 한다. 필요 시 맥박도 함께 입력할 수 있다.
- 혈당 입력, 복약 여부, 수면 시간·만족도, 운동 종류·시간, 컨디션 등은 선택 기능으로 제공한다.
- 식사와 물 섭취 기록은 MVP 필수 범위에서 제외하고, 확장 시 사진·즐겨찾기·간단 선택지 방식으로 제공한다.
- 기록은 가능한 한 적은 단계로 완료할 수 있어야 하며, 저장 후 수정·삭제할 수 있어야 한다.
- 누락된 날짜를 실패나 벌점으로 표시하지 않고, 과거 날짜 기록 보완과 기록 재개를 지원해야 한다.
- 음성 입력, 카메라 기반 입력, 건강 측정 기기 연동은 후속 검토 대상으로 한다.

### 2.4 대시보드 및 기록 조회
- 앱 첫 화면에는 오늘의 핵심 정보와 최근 기록 상태를 간단한 카드 형태로 제공해야 한다.
- 사용자는 날짜별 기록과 주간·월간 추이를 확인할 수 있어야 한다.
- 혈압·혈당과 같이 수동 입력한 값은 날짜별 목록과 그래프로 확인할 수 있어야 한다.
- 그래프는 색상뿐 아니라 텍스트와 기호를 함께 사용해 정보 의미를 전달해야 한다.
- 사용자는 원하는 기간과 지표를 선택해 상세 데이터를 볼 수 있어야 한다.
- 추세 요약에는 데이터 출처, 결측 여부, 측정 한계를 적절히 표시해야 한다.
- 복잡한 분석은 기본 화면을 방해하지 않도록 상세 화면에서 제공한다.

### 2.5 요약 및 개인화된 제안
- 앱은 사용자의 기록에서 관찰되는 경향을 쉬운 문장으로 요약할 수 있어야 한다.
- 제안은 한 번에 하나씩, 실행 가능한 작은 행동 중심으로 제공해야 한다.
- 사용자 목표는 현재 수준과 사용자가 선택한 목적을 고려해 설정·조정할 수 있어야 한다.
- 사용자는 제안을 거절하거나 목표를 직접 수정할 수 있어야 한다.
- 데이터가 부족하거나 측정값 간 차이가 큰 경우 확정적인 분석을 피하고 불확실성을 안내해야 한다.
- 수면·활동·심박 등 지표 간 관계는 상관관계를 인과관계나 의학적 결론처럼 표현해서는 안 된다.

### 2.6 알림 및 복약 관리
- 사용자는 알림 종류, 시간, 횟수, 소리·진동 여부를 조절하거나 알림을 끌 수 있어야 한다.
- 혈압·혈당 측정 및 복약 알림은 사용자가 설정한 일정에 따라 제공해야 한다.
- 알림에서 기록 완료, 복약 확인, 미루기 등 간단한 행동을 바로 수행할 수 있어야 한다.
- 알림은 사용자의 일정이 불규칙할 수 있음을 고려해 미루기·일시 중지 기능을 제공해야 한다.
- 일정 연동에 필요한 데이터와 권한이 확정되지 않은 경우, 근무표나 일정에 자동으로 맞춘다고 약속해서는 안 된다. 일정 기반 알림은 후속 기능으로 검토한다.
- 알림은 죄책감이나 압박을 주지 않는 문구를 사용해야 한다.

### 2.7 접근성 및 사용 편의성
- 글자 크기 조절을 지원하고, 주요 버튼과 입력란은 명확하고 충분히 큰 터치 영역을 제공해야 한다.
- 메뉴와 안내 문구는 쉬운 한국어를 사용하고, 한 화면에서 핵심 작업을 쉽게 찾을 수 있어야 한다.
- 기본 화면은 복잡한 통계보다 오늘의 상태와 다음 행동을 우선해서 보여줘야 한다.
- 오류나 잘못된 입력에 대해 수정·취소 방법을 안내해야 한다.
- 색상만으로 상태를 구분하지 않아야 하며, 화면 낭독 등 운영체제의 접근성 기능을 고려해야 한다.

### 2.8 개인정보 및 데이터 관리
- 사용자는 데이터 항목별 연동 권한을 확인하고 철회할 수 있어야 한다.
- 연동 해제 시 이후 동기화가 중단됨을 알리고, 이미 수집된 데이터의 처리 방식도 별도로 설명해야 한다.
- 사용자는 앱 안에서 선택한 기록 삭제, 전체 데이터 삭제 및 계정 삭제를 요청할 수 있어야 한다.
- 데이터 내보내기 기능은 MVP 범위에서 우선순위가 낮으며, 후속 버전에서 CSV 등 제공 방식을 검토한다.
- 제3자 제공은 기본적으로 비활성화하고, 선택형 외부 연동은 별도 동의를 받아야 한다.
- 건강 데이터의 수집 목적, 저장 위치, 보관 기간, 운영 위탁사 처리 여부 및 삭제 처리 기한을 개인정보 안내에 명시해야 한다.

### 2.9 선택형 사회·동기 부여 기능
- 친구·가족과의 공유, 챌린지, 배지 등은 사용자가 명시적으로 선택한 경우에만 제공해야 한다.
- 공유 전 누구에게 어떤 정보가 전달되는지 확인할 수 있어야 한다.
- MVP에서는 핵심 기록·조회·개인정보 기능보다 우선하지 않는다.

---

## 3. 비기능 요구사항

### 3.1 보안 및 개인정보 보호
- 건강 데이터 전송 구간은 TLS 등 적절한 암호화로 보호해야 한다.
- 서버에 저장하는 경우 암호화된 저장소와 최소 권한 원칙을 적용해야 한다.
- 인증 토큰 등 민감한 정보는 일반 데이터와 분리해 안전하게 관리해야 한다.
- 기기 내 저장은 운영체제의 보안 저장소를 활용해야 한다.
- 광고 목적으로 건강 데이터를 판매하거나 제공하지 않아야 한다.
- 운영을 위해 외부 업체가 데이터를 처리하는 경우 목적·범위와 보호 조치를 공개해야 한다.
- 데이터 접근과 변경은 서비스 운영에 필요한 범위로 제한해야 한다.

### 3.2 사용성 및 접근성
- 핵심 확인 및 기록 작업은 적은 화면 이동과 입력으로 완료할 수 있어야 한다.
- 큰 글씨, 명확한 버튼 이름, 일관된 메뉴 구성을 지원해야 한다.
- 알림과 기록은 건너뛰거나 나중에 수정할 수 있어야 한다.
- 기록이 비어 있는 날을 실패나 벌점으로 표현해서는 안 된다.

### 3.3 신뢰성 및 데이터 품질
- 연동 데이터의 출처, 시각, 단위와 동기화 상태를 확인할 수 있어야 한다.
- 오류·중복·결측 데이터를 임의로 정상화하지 않고, 처리 여부를 구분해야 한다.
- 동기화 실패 시 사용자가 데이터를 잘못된 최신 정보로 오인하지 않도록 상태를 표시해야 한다.
- 기기별 측정 차이와 센서 한계를 의료적 정확도로 오인하지 않도록 안내해야 한다.

### 3.4 성능 및 안정성
- 앱의 첫 화면과 최근 데이터 조회는 일상적으로 빠르게 사용할 수 있도록 설계해야 한다.
- 데이터 동기화가 지연되거나 실패해도 이미 저장된 기록을 확인할 수 있어야 한다.
- 네트워크 연결이 불안정한 환경에서 수동 기록이 유실되지 않도록 임시 저장 및 재시도 동작을 설계해야 한다.

### 3.5 투명성 및 의료 안전성
- 앱은 진단, 치료, 응급 판단을 제공하는 것처럼 표현해서는 안 된다.
- 건강 관련 요약과 제안은 참고 정보임을 명확히 안내해야 한다.
- 사용자가 걱정되는 증상이나 지속적으로 이상한 수치를 확인하는 경우 의료 전문가와 상담하도록 안내해야 한다.
- 제안의 근거와 한계를 사용자 수준에 맞는 말로 설명해야 한다.

### 3.6 정책 및 법률 준수
- 개인정보 처리, 동의, 보관 및 삭제는 출시 대상 국가·지역의 관련 법률과 플랫폼 정책을 검토한 뒤 확정해야 한다.
- 실제 서버 저장 여부, 보관 기간, 백업 및 삭제 처리 기한은 출시 전에 정책 문서와 앱 화면에 명시해야 한다.

---

## 4. 제약 조건

1. **플랫폼 권한 제약:** HealthKit 및 Health Connect가 제공하는 데이터와 사용자의 권한 허용 여부에 따라 자동 수집 가능한 항목이 달라진다.
2. **기기별 편차:** 웨어러블 및 측정 기기마다 제공 데이터, 측정 방법, 단위와 정확도가 다를 수 있다.
3. **연동 범위:** MVP는 iOS HealthKit과 Android Health Connect를 중심으로 하며, 제조사별 클라우드·SDK 및 외부 앱 직접 연동은 별도 검토가 필요하다.
4. **서버 저장 정책 미확정:** 서버 저장 여부, 데이터 보관 기간, 백업 및 삭제 처리 기한은 개발·보안·법률 검토 후 확정해야 한다.
5. **의료기기 범위 미포함:** 본 앱은 일상 건강 관리 및 기록 확인을 목적으로 하며, 별도 검토 없이 의료 진단·치료 기능을 표방하지 않는다.
6. **사용자 접근성 다양성:** 기술 숙련도와 시각·신체적 사용 조건이 다양하므로 설정 화면과 고급 분석을 기본 사용 흐름에 과도하게 포함하지 않는다.
7. **외부 서비스 의존성:** 외부 플랫폼 API 변경, 접근 제한, 서비스 중단에 따라 연동 기능이 영향을 받을 수 있다.
8. **일정 기반 알림 제약:** 근무표·캘린더 연동은 접근 권한과 서비스 연동 정책이 확정되기 전까지 MVP 필수 기능으로 간주하지 않는다.

---

## 5. 타깃 사용자

### 5.1 핵심 사용자군
- **자동 기록을 선호하는 사용자:** 스마트폰·웨어러블 데이터를 자동 수집하고 날짜별로 비교하고 싶은 사용자
- **혈압·혈당 등 특정 지표 관리 사용자:** 측정값과 복약 정보를 간단히 기록하고 추이를 확인하고 싶은 사용자
- **불규칙한 생활 사용자:** 교대근무, 프리랜서 업무, 학업 등으로 고정된 기록 시간과 알림이 맞지 않는 사용자
- **기술 숙련도가 낮거나 접근성 요구가 있는 사용자:** 큰 글씨, 쉬운 설명, 단순한 메뉴와 단계별 안내가 필요한 사용자
- **데이터 중심 사용자:** 출처, 동기화 상태, 세부 통계, 데이터 내보내기와 개인정보 설정을 중시하는 사용자

### 5.2 공통 사용자 특성
- 기록이 복잡하거나 알림이 잦으면 사용을 중단할 가능성이 높다.
- 자동 수집을 선호하지만, 누락된 데이터는 간단히 보완하고 싶어 한다.
- 건강 데이터가 어떻게 수집·저장·공유되는지 명확히 알기를 원한다.
- 의료적 단정 대신 이해하기 쉬운 요약과 현실적인 행동 제안을 선호한다.
- 기록을 놓친 날에도 부담 없이 다시 시작할 수 있기를 원한다.

---

## 6. 우선순위

| 우선순위 | 요구사항 |
|---|---|
| **P0 — 출시 필수** | iOS HealthKit·Android Health Connect 기본 연동 및 항목별 동의 |
| **P0 — 출시 필수** | 걸음 수·활동량·수면·심박수 등 지원 데이터의 자동 수집 및 출처·동기화 시각 표시 |
| **P0 — 출시 필수** | 날짜별 조회와 간결한 일간·주간 대시보드 |
| **P0 — 출시 필수** | 혈압 등 핵심 건강 수치의 간단한 수동 입력·수정·삭제 |
| **P0 — 출시 필수** | 알림 시간·빈도 조절, 미루기 및 끄기 |
| **P0 — 출시 필수** | 큰 글씨와 쉬운 문구, 누락 기록에 대한 비난 없는 안내 |
| **P0 — 출시 필수** | 연동 해제, 동의 철회, 데이터 삭제 및 계정 삭제 요청 기능 |
| **P0 — 출시 필수** | 암호화, 최소 수집, 제3자 제공 기본 비활성화, 개인정보 안내 |
| **P0 — 출시 필수** | 비진단 목적과 측정 한계 안내 |
| **P1 — 후속 핵심** | 혈당·복약·수면 만족도·컨디션 등 선택형 기록 및 측정 기기 연동 |
| **P1 — 후속 핵심** | 주·월 단위 비교, 목표 설정 및 개인 기준의 추세 요약 |
| **P1 — 후속 핵심** | 선택형 식사·물 기록, 사진·즐겨찾기 기반 간편 입력 |
| **P1 — 후속 핵심** | CSV 데이터 내보내기 및 데이터 관리 이력 |
| **P2 — 확장** | 제조사별 직접 연동, 운동·식단 외부 서비스 연동 |
| **P2 — 확장** | 일정·근무 패턴 기반 알림, 고급 분석 및 문서화된 API |
| **P2 — 확장** | 친구 공유, 선택형 챌린지, 배지·게임형 보상 |
| **P2 — 확장** | 음성 입력 및 고도화된 접근성 보조 기능 |

---

## 7. 위험과 완화 방안

| 위험 | 영향 | 완화 방안 |
|---|---|---|
| 기기·플랫폼별 데이터 차이와 측정 오차 | 사용자가 서로 다른 값을 잘못 해석할 수 있음 | 출처·측정 시각·단위·추정 여부를 표시하고, 의료적 정확도를 보장하지 않는다고 안내 |
| 데이터 중복 또는 날짜·시간대 오류 | 일별 통계가 부정확해질 수 있음 | 중복 처리 규칙을 적용하고 시간대 변경을 고려하며, 원본과 처리 상태를 확인할 수 있게 함 |
| 연동 권한 거부 또는 API 변경 | 자동 수집이 제한되거나 중단될 수 있음 | 권한별 영향과 재연결 방법을 설명하고, 필요 시 간편 수동 입력을 제공 |
| 데이터 누락 및 동기화 지연 | 최신 정보로 오인하거나 기록 흐름이 끊길 수 있음 | 마지막 동기화 시각과 누락 상태를 표시하고, 지연·오류 메시지를 명확히 제공 |
| 건강 데이터 유출 또는 무단 접근 | 사용자 피해와 서비스 신뢰 저하 | 최소 수집, 암호화, 접근 권한 제한, 토큰 분리 보관, 보안 점검 및 대응 절차 마련 |
| 삭제 범위에 대한 오해 | 사용자가 연동 해제를 데이터 완전 삭제로 오인할 수 있음 | 연동 해제와 서버 데이터 삭제를 구분해 설명하고, 삭제 범위·처리 기한·플랫폼 측 삭제 방법을 안내 |
| 분석 결과가 의료 조언으로 오인됨 | 부적절한 건강 판단이나 치료 지연 가능성 | 진단처럼 단정하지 않고, 분석 근거와 불확실성을 표시하며 필요 시 전문가 상담을 권고 |
| 잦거나 부적절한 알림 | 사용자가 알림을 끄거나 앱을 이탈할 수 있음 | 시간·빈도·종류를 조절하게 하고, 미루기·일시 중지·완전 끄기를 쉽게 제공 |
| 고령 사용자 또는 낮은 기술 숙련도의 사용 어려움 | 초기 설정과 기록 완료율이 낮아질 수 있음 | 큰 글씨·단순 메뉴·단계별 안내·쉬운 용어를 제공하고, 오류 수정 방법을 명확히 안내 |
| 개인화 분석의 근거 부족 | 잘못된 패턴이나 과도한 확신을 제시할 수 있음 | 데이터 부족 시 분석을 보류하고 확신 수준을 표시하며, 상관관계를 인과관계로 표현하지 않음 |
| 외부 서비스 연동 범위 확대에 따른 일정·운영 복잡성 | 개발 비용과 유지보수 부담 증가 | MVP에서 표준 건강 플랫폼을 우선하고, 외부 서비스는 수요·접근 권한·보안 검토 후 단계적으로 추가 |
| 광고·유료화에 대한 불신 | 사용자의 이탈 또는 개인정보 우려 증가 | 기본 기록·요약 기능의 제공 범위와 유료 기능을 명확히 설명하고, 자동 결제·해지 절차를 투명하게 제공 |